# Title 3: Protein and energy delivery vs ASPEN/SCCM and ESPEN targets
**Critically ill patients with acute or chronic kidney disease, MIMIC-III v1.4 (MetaVision 2008-2012)**

**Data rules (PhysioNet DUA)**
- Run this notebook only on your own computer. Never upload MIMIC files, `local_*` files or screenshots of rows to any online service, including chat assistants.
- Files that start with `summary_` contain only aggregate numbers; counts from 1 to 10 are shown as `<11`. Only these may be shared.

**Files needed in the same folder as this notebook**

| Already downloaded | Download now |
|---|---|
| ICUSTAYS, PATIENTS, ADMISSIONS, DIAGNOSES_ICD, PROCEDURES_ICD | INPUTEVENTS_MV, PROCEDUREEVENTS_MV, D_ITEMS |

```bash
for f in INPUTEVENTS_MV PROCEDUREEVENTS_MV D_ITEMS; do
  wget -N -c --user USERNAME --ask-password https://physionet.org/files/mimiciii/1.4/$f.csv.gz
done
```
Put everything (MIMIC `.csv.gz` files, `mimic_t3.py`, `formula_composition.csv`, `guideline_targets.csv`) in `C:\Eksperimen\mimic\gizi`. Results are written to `C:\Eksperimen\mimic\gizi\outputs`.

In [ ]:
import os, sys
from pathlib import Path
# All MIMIC files, the .py modules and the .csv tables are in C:\Eksperimen\mimic\gizi
# (seen from WSL as /mnt/c/Eksperimen/mimic/gizi).
CANDIDATES = [os.environ.get('MIMIC_DIR', ''), '/mnt/c/Eksperimen/mimic/gizi', r'C:\Eksperimen\mimic\gizi']
DATA = next((Path(p) for p in CANDIDATES if p and Path(p).is_dir()), None)
assert DATA is not None, r'Folder C:\Eksperimen\mimic\gizi not found; check the path'
sys.path.insert(0, str(DATA))            # so that mimic_t3.py / mimic_t1.py are found
OUT = DATA / 'outputs'; OUT.mkdir(exist_ok=True)
print('Data folder:', DATA)
print('Output folder:', OUT)
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import mimic_t3 as m
need = ['ICUSTAYS','PATIENTS','ADMISSIONS','DIAGNOSES_ICD','PROCEDURES_ICD',
        'INPUTEVENTS_MV','PROCEDUREEVENTS_MV','D_ITEMS']
missing = [f for f in need if not (DATA / f'{f}.csv.gz').exists()]
print('Missing files:', missing or 'none')

## Step 1. Cohort and kidney groups

In [ ]:
icu = m.load_icustays(DATA)
cohort, flow = m.base_cohort(icu)
dx = m.kidney_dx(DATA, cohort.HADM_ID)
pf, proc_labels = m.procedure_flags(DATA, cohort)
cohort = cohort.merge(dx, on='HADM_ID', how='left').merge(pf, on='ICUSTAY_ID', how='left')
for c in ['AKI','CKD','ESRD','KRT_ICD','CKRT_EVT','IKRT_EVT','VENT']:
    cohort[c] = cohort[c].fillna(False).astype(bool)
cohort = m.assign_group(cohort)
print(proc_labels.to_string(index=False))      # dictionary labels used for KRT and ventilation
cohort.GROUP.value_counts()

## Step 2. Nutrition items and composition

In [ ]:
comp = pd.read_csv(DATA / 'formula_composition.csv')
targets = pd.read_csv(DATA / 'guideline_targets.csv')
items = m.nutrition_items(DATA, comp)
print(len(items), 'candidate items;', items.type.notna().sum(), 'mapped to the composition table')

In [ ]:
# One pass over INPUTEVENTS_MV (about 3.6 million rows); takes a few minutes.
rows, weights = m.scan_inputevents(DATA, cohort, items)
r = m.nutrient_amounts(rows, items)
inv = m.item_inventory(items, r)
m.save_summary(inv, OUT, 'summary_items')
print('Rows with an unexpected unit:', int((~r.UNIT_OK & r.type.isin(['EN','MODULAR','PROPOFOL'])).sum()))
inv.head(40)

### Check the item list before continuing
- Items with `mapped = False` or blank `kcal_per_unit`/`protein_g_per_unit` contribute **no** nutrients.
- Add or correct rows in `formula_composition.csv` using the manufacturer label (public information, so Bu A may check it), then re-run from Step 2.
- `summary_items.csv` holds only dictionary labels and counts, so it may be shared for help with the mapping.

## Step 3. Daily delivery and final cohort

In [ ]:
alloc = m.allocate_days(r, cohort)
stay = alloc.groupby('ICUSTAY_ID').agg(EN_ML=('EN_ML','sum'), PN_ML=('PN_ML','sum')).reset_index()
first_en = alloc[alloc.EN_ML > 0].groupby('ICUSTAY_ID').DAY.min().rename('FIRST_EN_DAY').reset_index()
cohort = (cohort.drop(columns=[c for c in ['WEIGHT','EN_ML','PN_ML','FIRST_EN_DAY'] if c in cohort])
          .merge(weights, on='ICUSTAY_ID', how='left').merge(stay, on='ICUSTAY_ID', how='left')
          .merge(first_en, on='ICUSTAY_ID', how='left'))
cohort[['EN_ML','PN_ML']] = cohort[['EN_ML','PN_ML']].fillna(0)

f = flow.copy()
c1 = cohort[cohort.EN_ML > 0];                 f.append(('Received enteral nutrition on ICU days 1-7', len(c1), c1.SUBJECT_ID.nunique()))
pn_users = c1[c1.PN_ML > 0]
c2 = c1[c1.PN_ML == 0];                        f.append(('No parenteral nutrition on ICU days 1-7', len(c2), c2.SUBJECT_ID.nunique()))
c3 = c2[c2.WEIGHT.between(30, 250)];           f.append(('Recorded weight 30-250 kg', len(c3), c3.SUBJECT_ID.nunique()))
final = c3.reset_index(drop=True)
flow_df = pd.DataFrame(f, columns=['step','icu_stays','patients'])
flow_df[['icu_stays','patients']] = flow_df[['icu_stays','patients']].map(m.suppress)
m.save_summary(flow_df, OUT, 'summary_flow')
flow_df

In [ ]:
day = m.daily_table(final, alloc)
means = m.window_means(day)                    # ICU days 4-7
t1 = m.table1(final, means)
m.save_summary(t1, OUT, 'summary_table1')
t1.T

## Step 4. Delivery by day and guideline adequacy

In [ ]:
daily = m.daily_summary(day)
m.save_summary(daily, OUT, 'summary_daily')
adq = m.adequacy(means, final, targets)
m.save_summary(adq, OUT, 'summary_adequacy')
adq

In [ ]:
# Kidney disease without KRT, split into AKI only and CKD (targets of the parent group)
kid = final[final.GROUP == 'Kidney disease without KRT']
sub = m.adequacy(means, kid, targets, group_col='SUBGROUP',
                 target_map={'AKI only': 'Kidney disease without KRT',
                             'CKD (with or without AKI)': 'Kidney disease without KRT'})
m.save_summary(sub, OUT, 'summary_subgroup')
sub

## Step 5. Factors associated with protein delivery (days 4-7)

In [ ]:
reg = m.regression(final, means)
reg.attrs['n'] = m.suppress(reg.attrs['n'])
m.save_summary(reg.assign(n=reg.attrs['n']), OUT, 'summary_regression')
reg

## Step 6. Sensitivity analyses

In [ ]:
def reach(means_, cohort_, label, energy_col='KCAL_KG'):
    a = m.adequacy(means_, cohort_, targets, energy_col=energy_col)
    keep = ['guideline','group','n','protein_reach80_pct','protein_within_pct','energy_reach80_pct','energy_within_pct']
    return a.reindex(columns=keep).assign(analysis=label)

sens = [reach(means, final, 'Primary: days 4-7, all full ICU days'),
        reach(m.window_means(day, (1, 7)), final, 'Days 1-7'),
        reach(m.window_means(day, fed_only=True), final, 'Days 4-7, fed days only'),
        reach(means, final[final.WEIGHT <= 120], 'Weight 120 kg or less'),
        reach(means, final, 'Energy including propofol', energy_col='KCAL_TOTAL_KG')]
# Including patients who also received PN; PN amino acids, dextrose and lipids are
# recorded in grams in MetaVision and are added to enteral delivery here.
def enpn(means_):
    return means_.assign(PROTEIN_GKG=means_.PROTEIN_ENPN_GKG, KCAL_KG=means_.KCAL_ENPN_KG)
pn_ok = pn_users[pn_users.WEIGHT.between(30, 250)]
if len(pn_ok):
    sens.append(reach(enpn(m.window_means(m.daily_table(pn_ok, alloc))), pn_ok, 'PN users only, EN + PN'))
all_fed = pd.concat([final, pn_ok], ignore_index=True)
sens.append(reach(enpn(m.window_means(m.daily_table(all_fed, alloc))), all_fed, 'All EN recipients incl. PN users, EN + PN'))
sens = pd.concat(sens, ignore_index=True)
m.save_summary(sens, OUT, 'summary_sensitivity')
sens

## Step 7. Figures (aggregate only)

In [ ]:
COL = {'No kidney disease':'#0072B2','Kidney disease without KRT':'#E69F00','Intermittent KRT':'#009E73','CKRT':'#CC79A7'}
d = daily.dropna(subset=['protein_gkg_median'])
fig, ax = plt.subplots(1, 2, figsize=(11, 4), sharex=True)
for g, s in d.groupby('GROUP', observed=True):
    ax[0].errorbar(s.DAY, s.protein_gkg_median, yerr=[s.protein_gkg_median-s.protein_gkg_q1, s.protein_gkg_q3-s.protein_gkg_median],
                   marker='o', ms=4, lw=2, capsize=2, color=COL[g], label=g)
    ax[1].errorbar(s.DAY, s.kcal_kg_median, yerr=[s.kcal_kg_median-s.kcal_kg_q1, s.kcal_kg_q3-s.kcal_kg_median],
                   marker='o', ms=4, lw=2, capsize=2, color=COL[g], label=g)
ax[0].axhspan(1.2, 2.0, color='0.9', zorder=0); ax[0].text(1, 1.95, 'ASPEN/SCCM 1.2-2.0', fontsize=8, va='top')
ax[1].axhspan(12, 25, color='0.9', zorder=0); ax[1].text(1, 24.5, 'ASPEN/SCCM 12-25', fontsize=8, va='top')
ax[0].set(xlabel='ICU day', ylabel='Protein delivered (g/kg/day), median and IQR')
ax[1].set(xlabel='ICU day', ylabel='Energy delivered (kcal/kg/day), median and IQR')
for a in ax: a.spines[['top','right']].set_visible(False)
ax[0].legend(frameon=False, fontsize=8)
fig.tight_layout(); fig.savefig(OUT/'summary_fig1_daily_delivery.png', dpi=300)

In [ ]:
a = adq.dropna(subset=['protein_below_pct'])
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True)
for ax, gl in zip(axes, ['ASPEN/SCCM', 'ESPEN']):
    s = a[a.guideline == gl].set_index('group').reindex([g for g in m.GROUPS if g in set(a.group)])
    left = np.zeros(len(s))
    for col, lab, colr in [('protein_below_pct','Below','#D55E00'),('protein_within_pct','Within','#009E73'),('protein_above_pct','Above','#56B4E9')]:
        ax.barh(s.index, s[col], left=left, color=colr, label=lab, edgecolor='white', linewidth=2)
        left += s[col].to_numpy()
    ax.set(title=f'{gl} protein target', xlabel='% of ICU stays (mean of days 4-7)', xlim=(0, 100))
    ax.spines[['top','right']].set_visible(False)
axes[0].legend(frameon=False, fontsize=8, loc='lower right')
fig.tight_layout(); fig.savefig(OUT/'summary_fig2_protein_adequacy.png', dpi=300)

## Step 8. Link: early dietitian assessment and delivery on days 4-7
Needs `NOTEEVENTS.csv.gz` and `mimic_t1.py` in the same folder. Only note **times** are read; the TEXT column is never loaded.
If the kernel was restarted, run all cells above first (*Kernel -> Restart Kernel and Run All Cells*).

In [ ]:
import mimic_t1 as m1
notes = m1.nutrition_note_times(DATA, final.HADM_ID)        # several minutes
linked, link = m1.link_assessment_delivery(final, means, notes)
m.save_summary(link, OUT, 'summary_link_delivery')
reg_link = m.regression(linked.drop(columns=['PROTEIN_GKG', 'KCAL_KG']), means,
                        extra={'Dietitian assessment within 72 h': 'EARLY_ASSESS'})
reg_link['n'] = m.suppress(len(linked))
m.save_summary(reg_link, OUT, 'summary_link_regression')
print('Assessed within 72 h:', m.suppress(int(linked.EARLY_ASSESS.sum())), 'of', m.suppress(len(linked)))
link

## What to send
Send only the files in `outputs/` whose names start with `summary_`:
`summary_flow`, `summary_items`, `summary_table1`, `summary_daily`, `summary_adequacy`, `summary_subgroup`,
`summary_regression`, `summary_sensitivity`, `summary_link_delivery`, `summary_link_regression` (.csv) and the two `summary_fig*.png` figures.

In [ ]:
sorted(p.name for p in OUT.iterdir())